In [130]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

In [131]:
X = pd.read_csv('../src/data/raw/dengue_features_train.csv')
y = pd.read_csv('../src/data/raw/dengue_labels_train.csv')
y = y[['total_cases']]

In [132]:
# Describe the data
print(X.describe())

              year   weekofyear      ndvi_ne      ndvi_nw      ndvi_se  \
count  1456.000000  1456.000000  1262.000000  1404.000000  1434.000000   
mean   2001.031593    26.503434     0.142294     0.130553     0.203783   
std       5.408314    15.019437     0.140531     0.119999     0.073860   
min    1990.000000     1.000000    -0.406250    -0.456100    -0.015533   
25%    1997.000000    13.750000     0.044950     0.049217     0.155087   
50%    2002.000000    26.500000     0.128817     0.121429     0.196050   
75%    2005.000000    39.250000     0.248483     0.216600     0.248846   
max    2010.000000    53.000000     0.508357     0.454429     0.538314   

           ndvi_sw  precipitation_amt_mm  reanalysis_air_temp_k  \
count  1434.000000           1443.000000            1446.000000   
mean      0.202305             45.760388             298.701852   
std       0.083903             43.715537               1.362420   
min      -0.063457              0.000000             294.635714  

In [133]:
# Check for missing values
print(X.isnull().sum())

city                                       0
year                                       0
weekofyear                                 0
week_start_date                            0
ndvi_ne                                  194
ndvi_nw                                   52
ndvi_se                                   22
ndvi_sw                                   22
precipitation_amt_mm                      13
reanalysis_air_temp_k                     10
reanalysis_avg_temp_k                     10
reanalysis_dew_point_temp_k               10
reanalysis_max_air_temp_k                 10
reanalysis_min_air_temp_k                 10
reanalysis_precip_amt_kg_per_m2           10
reanalysis_relative_humidity_percent      10
reanalysis_sat_precip_amt_mm              13
reanalysis_specific_humidity_g_per_kg     10
reanalysis_tdtr_k                         10
station_avg_temp_c                        43
station_diur_temp_rng_c                   43
station_max_temp_c                        20
station_mi

In [134]:
import sys
sys.path.append('../src')

In [135]:
from transformers.transformers_l import DropColumnsTransformer


In [136]:
# Instantiate the transformer with the columns to drop
drop_columns_transformer = DropColumnsTransformer(
    columns_to_drop=['ndvi_ne', 'ndvi_nw', 'station_avg_temp_c', 'station_diur_temp_rng_c', 'week_start_date']
)

# Apply the transformer to the DataFrame
X = drop_columns_transformer.fit_transform(X)

In [137]:

# # Drop the columns with too many missing values
# X.drop(columns=['ndvi_ne', 'ndvi_nw', 'station_avg_temp_c', 'station_diur_temp_rng_c'], inplace=True)

# # Drop date columns
# X.drop(columns=['week_start_date'], inplace=True)

# Transform categorical values in the 'city' column to numerical values
X['city'] = X['city'].map({'sj': 0, 'iq': 1})

# Check for missing values
print(X.isnull().sum())


city                                      0
year                                      0
weekofyear                                0
ndvi_se                                  22
ndvi_sw                                  22
precipitation_amt_mm                     13
reanalysis_air_temp_k                    10
reanalysis_avg_temp_k                    10
reanalysis_dew_point_temp_k              10
reanalysis_max_air_temp_k                10
reanalysis_min_air_temp_k                10
reanalysis_precip_amt_kg_per_m2          10
reanalysis_relative_humidity_percent     10
reanalysis_sat_precip_amt_mm             13
reanalysis_specific_humidity_g_per_kg    10
reanalysis_tdtr_k                        10
station_max_temp_c                       20
station_min_temp_c                       14
station_precip_mm                        22
dtype: int64


In [138]:
X.head()

,city,year,weekofyear,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,reanalysis_avg_temp_k,reanalysis_dew_point_temp_k,reanalysis_max_air_temp_k,reanalysis_min_air_temp_k,reanalysis_precip_amt_kg_per_m2,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_max_temp_c,station_min_temp_c,station_precip_mm
0,0,1990,18,0.198483,0.177617,12.42,297.572857,297.742857,292.414286,299.8,295.9,32.00,73.365714,12.42,14.012857,2.628571,29.4,20.0,16.0
1,0,1990,19,0.162357,0.155486,22.82,298.211429,298.442857,293.951429,300.9,296.4,17.94,77.368571,22.82,15.372857,2.371429,31.7,22.2,8.6
2,0,1990,20,0.157200,0.170843,34.54,298.781429,298.878571,295.434286,300.5,297.3,26.10,82.052857,34.54,16.848571,2.300000,32.2,22.8,41.4
3,0,1990,21,0.227557,0.235886,15.36,298.987143,299.228571,295.310000,301.4,297.0,13.90,80.337143,15.36,16.672857,2.428571,33.3,23.3,4.0
4,0,1990,22,0.251200,0.247340,7.52,299.518571,299.664286,295.821429,301.9,297.5,12.20,80.460000,7.52,17.210000,3.014286,35.0,23.9,5.8


In [139]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Identify columns with missing values in the training set
columns_with_missing = X_train.columns[X_train.isnull().any()]

# Check if there are columns with missing values
if not columns_with_missing.empty:
    # Create a SimpleImputer instance with strategy='mean'
    imputer = SimpleImputer(strategy='mean')

    # Fit the imputer on the training set and transform both training and testing sets
    X_train[columns_with_missing] = imputer.fit_transform(X_train[columns_with_missing])
    X_test[columns_with_missing] = imputer.transform(X_test[columns_with_missing])

# Check the result
print(X_train.isnull().sum())
print(X_test.isnull().sum())

city                                     0
year                                     0
weekofyear                               0
ndvi_se                                  0
ndvi_sw                                  0
precipitation_amt_mm                     0
reanalysis_air_temp_k                    0
reanalysis_avg_temp_k                    0
reanalysis_dew_point_temp_k              0
reanalysis_max_air_temp_k                0
reanalysis_min_air_temp_k                0
reanalysis_precip_amt_kg_per_m2          0
reanalysis_relative_humidity_percent     0
reanalysis_sat_precip_amt_mm             0
reanalysis_specific_humidity_g_per_kg    0
reanalysis_tdtr_k                        0
station_max_temp_c                       0
station_min_temp_c                       0
station_precip_mm                        0
dtype: int64
city                                     0
year                                     0
weekofyear                               0
ndvi_se                                  

In [140]:
y_train.describe()

,total_cases
count,1164.000000
mean,24.608247
std,43.127312
min,0.000000
25%,5.000000
50%,12.000000
75%,28.000000
max,461.000000


In [141]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)

In [142]:

y_train.head()

,total_cases
1399,2
1308,1
679,7
638,4
247,46


In [143]:
# Fit the model
model.fit(X_train, y_train)
# Make predictions
y_pred = model.predict(X_test)
# Calculate the mean absolute error
mean_absolute_error(y_test, y_pred)

/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


13.55263698630137

In [144]:
# Apply simple linear regression
from sklearn.linear_model import LinearRegression

lnmodel = LinearRegression()
# Fit the model
lnmodel.fit(X_train, y_train)
# Make predictions
y_pred = lnmodel.predict(X_test)
# Calculate the mean absolute error
mean_absolute_error(y_test, y_pred)

22.170991577112932

In [145]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(random_state=0),
    "Lasso Regression": Lasso(random_state=0),
    "Decision Tree": DecisionTreeRegressor(random_state=0),
    "Random Forest": RandomForestRegressor(random_state=0)
}

# ------------------------------
# Evaluate Models with Cross Validation
# ------------------------------

results_list = []

for model_name, model in models.items():
    # Build a pipeline combining preprocessing and the regressor
    pipeline = Pipeline(steps=[
        ('regressor', model)
    ])
    
    # Perform cross validation with 5 folds
    # Note: Scoring returns negative values for MAE and RMSE, so convert them back.
    cv_results = cross_validate(
        pipeline, X_train, y_train,
        cv=5,
        scoring={
            'MAE': 'neg_mean_absolute_error',
            'RMSE': 'neg_root_mean_squared_error',
            'R2': 'r2'
        },
        return_train_score=False
    )
    
    mae = -np.mean(cv_results['test_MAE'])
    rmse = -np.mean(cv_results['test_RMSE'])
    r2 = np.mean(cv_results['test_R2'])
    
    results_list.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# Convert results to a DataFrame for nicer display
results_df = pd.DataFrame(results_list)
print(results_df)

/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please ch

               Model        MAE       RMSE        R2
0  Linear Regression  19.956854  38.507909  0.176155
1   Ridge Regression  19.895139  38.489836  0.176659
2   Lasso Regression  19.796183  38.513249  0.175248
3      Decision Tree  16.745220  39.824713  0.080825
4      Random Forest  13.517325  27.635619  0.570276


In [146]:
# Load the test features 
X_test_competition  = pd.read_csv('../src/data/raw/dengue_features_test.csv')

X_test_competition

,city,year,weekofyear,week_start_date,ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,...,reanalysis_precip_amt_kg_per_m2,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm
0,sj,2008,18,2008-04-29,-0.018900,-0.018900,0.102729,0.091200,78.60,298.492857,...,25.37,78.781429,78.60,15.918571,3.128571,26.528571,7.057143,33.3,21.7,75.2
1,sj,2008,19,2008-05-06,-0.018000,-0.012400,0.082043,0.072314,12.56,298.475714,...,21.83,78.230000,12.56,15.791429,2.571429,26.071429,5.557143,30.0,22.2,34.3
2,sj,2008,20,2008-05-13,-0.001500,NaN,0.151083,0.091529,3.66,299.455714,...,4.12,78.270000,3.66,16.674286,4.428571,27.928571,7.785714,32.8,22.8,3.0
3,sj,2008,21,2008-05-20,NaN,-0.019867,0.124329,0.125686,0.00,299.690000,...,2.20,73.015714,0.00,15.775714,4.342857,28.057143,6.271429,33.3,24.4,0.3
4,sj,2008,22,2008-05-27,0.056800,0.039833,0.062267,0.075914,0.76,299.780000,...,4.36,74.084286,0.76,16.137143,3.542857,27.614286,7.085714,33.3,23.3,84.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,iq,2013,22,2013-05-28,0.301471,0.380029,0.280629,0.383186,41.12,297.774286,...,67.60,89.990000,41.12,17.185714,10.100000,27.400000,9.050000,32.6,21.8,33.0
412,iq,2013,23,2013-06-04,0.247600,0.296343,0.285371,0.350357,71.52,297.167143,...,45.70,93.891429,71.52,17.448571,9.657143,27.520000,10.720000,33.8,21.4,68.0
413,iq,2013,24,2013-06-11,0.238729,0.251029,0.252586,0.249771,78.96,295.831429,...,45.22,94.967143,78.96,16.410000,7.385714,27.200000,10.075000,32.6,21.6,93.2
414,iq,2013,25,2013-06-18,0.310429,0.302700,0.406614,0.403943,39.54,295.778571,...,4.70,89.057143,39.54,15.137143,8.228571,26.700000,8.480000,32.2,21.8,34.1


In [147]:
X_test_competition  = pd.read_csv('../src/data/raw/dengue_features_test.csv')
#print column names
print(X_test_competition)

    city  year  weekofyear week_start_date   ndvi_ne   ndvi_nw   ndvi_se  \
0     sj  2008          18      2008-04-29 -0.018900 -0.018900  0.102729   
1     sj  2008          19      2008-05-06 -0.018000 -0.012400  0.082043   
2     sj  2008          20      2008-05-13 -0.001500       NaN  0.151083   
3     sj  2008          21      2008-05-20       NaN -0.019867  0.124329   
4     sj  2008          22      2008-05-27  0.056800  0.039833  0.062267   
..   ...   ...         ...             ...       ...       ...       ...   
411   iq  2013          22      2013-05-28  0.301471  0.380029  0.280629   
412   iq  2013          23      2013-06-04  0.247600  0.296343  0.285371   
413   iq  2013          24      2013-06-11  0.238729  0.251029  0.252586   
414   iq  2013          25      2013-06-18  0.310429  0.302700  0.406614   
415   iq  2013          26      2013-06-25  0.339467  0.240071  0.356943   

      ndvi_sw  precipitation_amt_mm  reanalysis_air_temp_k  ...  \
0    0.091200       

In [148]:
# Check values for X_test_competition['city']
print(X_test_competition['city'].unique())

['sj' 'iq']


In [149]:
model.fit(X_train, y_train)

# Check if the columns exist in the test set ['ndvi_ne', 'ndvi_nw', 'station_avg_temp_c', 'station_diur_temp_rng_c', 'week_start_date'] before dropping
if 'ndvi_ne' in X_test_competition.columns:
    X_test_competition.drop(columns=['ndvi_ne', 'ndvi_nw', 'station_avg_temp_c', 'station_diur_temp_rng_c', 'week_start_date'], inplace=True)

# Check if the 'city' column has string values before mapping
if X_test_competition['city'].dtype == 'object':
    # Transform categorical values in the 'city' column to numerical values
    X_test_competition['city'] = X_test_competition['city'].map({'sj': 0, 'iq': 1})

# Handle missing values using the same imputer
columns_with_missing = X_test_competition.columns[X_test_competition.isnull().any()]
if not columns_with_missing.empty:
    X_test_competition[columns_with_missing] = imputer.transform(X_test_competition[columns_with_missing])

# Make predictions
predictions = model.predict(X_test_competition)


/opt/anaconda3/envs/oop/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [150]:
X_test_competition

,city,year,weekofyear,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,reanalysis_avg_temp_k,reanalysis_dew_point_temp_k,reanalysis_max_air_temp_k,reanalysis_min_air_temp_k,reanalysis_precip_amt_kg_per_m2,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_max_temp_c,station_min_temp_c,station_precip_mm
0,0,2008,18,0.102729,0.091200,78.60,298.492857,298.550000,294.527143,301.1,296.4,25.37,78.781429,78.60,15.918571,3.128571,33.3,21.7,75.2
1,0,2008,19,0.082043,0.072314,12.56,298.475714,298.557143,294.395714,300.8,296.7,21.83,78.230000,12.56,15.791429,2.571429,30.0,22.2,34.3
2,0,2008,20,0.151083,0.091529,3.66,299.455714,299.357143,295.308571,302.2,296.4,4.12,78.270000,3.66,16.674286,4.428571,32.8,22.8,3.0
3,0,2008,21,0.124329,0.125686,0.00,299.690000,299.728571,294.402857,303.0,296.9,2.20,73.015714,0.00,15.775714,4.342857,33.3,24.4,0.3
4,0,2008,22,0.062267,0.075914,0.76,299.780000,299.671429,294.760000,302.3,297.3,4.36,74.084286,0.76,16.137143,3.542857,33.3,23.3,84.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,1,2013,22,0.280629,0.383186,41.12,297.774286,298.964286,295.638571,305.5,292.7,67.60,89.990000,41.12,17.185714,10.100000,32.6,21.8,33.0
412,1,2013,23,0.285371,0.350357,71.52,297.167143,298.328571,295.845714,306.3,291.6,45.70,93.891429,71.52,17.448571,9.657143,33.8,21.4,68.0
413,1,2013,24,0.252586,0.249771,78.96,295.831429,296.607143,294.894286,304.6,290.7,45.22,94.967143,78.96,16.410000,7.385714,32.6,21.6,93.2
414,1,2013,25,0.406614,0.403943,39.54,295.778571,297.400000,293.648571,305.9,292.5,4.70,89.057143,39.54,15.137143,8.228571,32.2,21.8,34.1


In [151]:
# round the predictions to the nearest integer
predictions = np.round(predictions).astype(int)

# Create a DataFrame for the predictions as city,year,weekofyear,total_cases
predictions_df = pd.DataFrame({
    'city': X_test_competition['city'],
    'year': X_test_competition['year'],
    'weekofyear': X_test_competition['weekofyear'],
    'total_cases': predictions
})

# Revert the mapping for city
predictions_df['city'] = predictions_df['city'].map({0: 'sj', 1: 'iq'})

# Show predictions
print(predictions_df.head())

# Save the predictions to a CSV file
predictions_df.to_csv('../src/data/predictions/baseline_prediction.csv', index=False)

  city  year  weekofyear  total_cases
0   sj  2008          18            4
1   sj  2008          19            4
2   sj  2008          20            6
3   sj  2008          21            7
4   sj  2008          22            8
